# Praktikum 11 – Modell-Serving und Deployment-Profile
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Modelfiles gezielt konfigurieren, einen FastAPI-Wrapper real testen und Latenz mit Speicherplanung fuer verschiedene Deployment-Profile verbinden.

**Zielprodukt nach 90 Minuten:**
1. Zwei unterschiedliche Modellprofile aus Modelfiles.
2. Ein lauffaehiger API-Request-Zyklus per FastAPI-TestClient.
3. Eine kleine Benchmark- und Speicherplanungstabelle fuer Deployment-Entscheidungen.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
    "fastapi": ("fastapi", "0.115.12"),
    "uvicorn": ("uvicorn", "0.34.2"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv ist erforderlich. Bitte installiere uv und fuehre die Zelle erneut aus.")

    commands = [
        ["uv", "pip", "install", "--python", sys.executable, *specs],
    ]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        commands = [command + ["--system"] for command in commands]

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
if MODEL not in available_models:
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Zwei Modelfile-Profile vergleichen (25 min)
Wir erzeugen zwei Profile desselben Basismodells und testen sie am gleichen Codebeispiel.

**Pflichtschritte:**
- Lege zwei unterschiedliche Modelfiles an.
- Registriere beide Profile mit `ollama create`.
- Vergleiche danach die Antworten am selben Eingabetext.

**Soll-Ergebnis:** sichtbare Unterschiede zwischen striktem und erklaerendem Profil.

In [ ]:
PROFILE_DEFINITIONS = {
    "code-reviewer-strict": f"""FROM {MODEL}
SYSTEM Du bist ein strenger Python-Code-Reviewer. Antworte knapp in maximal drei Stichpunkten.
PARAMETER num_ctx 2048
PARAMETER temperature 0.0
""",
    "code-reviewer-explainer": f"""FROM {MODEL}
SYSTEM Du bist ein hilfsbereiter Python-Code-Reviewer. Erklaere kurz Problem, Risiko und Verbesserung.
PARAMETER num_ctx 4096
PARAMETER temperature 0.2
""",
}

for profile_name, profile_config in PROFILE_DEFINITIONS.items():
    modelfile_path = f"{profile_name}.Modelfile"
    with open(modelfile_path, "w", encoding="utf-8") as handle:
        handle.write(profile_config)
    print(f"Registriere Modellprofil {profile_name}...")
    subprocess.check_call(["ollama", "create", profile_name, "-f", modelfile_path])

code_sample = "def add(a, b): return a+b  # keine Typen, keine Dokumentation"
for profile_name in PROFILE_DEFINITIONS:
    response = ollama.chat(
        model=profile_name,
        messages=[{"role": "user", "content": code_sample}],
        think=False,
        options={"num_predict": 120},
        keep_alive="10m",
    )["message"]["content"].strip()
    print(f"\nProfil: {profile_name}")
    print(response)

## Teil 2 – FastAPI-Wrapper mit echtem Request-Zyklus (30 min)
Wir testen den API-Weg nicht nur konzeptionell, sondern mit einem echten Request ueber den TestClient.

**Pflichtschritte:**
- Baue eine Route, die Eingaben vorverarbeitet.
- Messe die Laufzeit eines API-Calls.
- Gib neben der Antwort auch Nutzungsmetadaten zurueck.

**Soll-Ergebnis:** eine funktionierende lokale API mit nachvollziehbarer Antwortstruktur.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
import re

EMAIL_PATTERN = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_PATTERN = re.compile(r"\+?\d[\d\s\-/]{7,}\d")


def sanitize_prompt(prompt):
    sanitized = EMAIL_PATTERN.sub("[EMAIL]", prompt)
    sanitized = PHONE_PATTERN.sub("[TELEFON]", sanitized)
    return sanitized


app = FastAPI()


@app.get("/generate")
def generate(prompt: str, temperature: float = 0.0):
    safe_prompt = sanitize_prompt(prompt)
    started = time.perf_counter()
    response = ollama.generate(
        model=MODEL,
        prompt=safe_prompt,
        options={"temperature": temperature},
    )
    latency_s = round(time.perf_counter() - started, 2)
    answer = response["response"].strip()
    if not answer:
        raise RuntimeError("The model returned an empty response.")
    return {
        "sanitized_prompt": safe_prompt,
        "response": answer,
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "latency_s": latency_s,
    }


client = TestClient(app)
api_response = client.get(
    "/generate",
    params={
        "prompt": "Schreibe eine kurze Begruessung an max@example.de und erklaere in einem Satz RAG.",
        "temperature": 0.0,
    },
)
print(api_response.json())

## Teil 3 – Serving-Benchmark und Speicherplanung (35 min)
Wir messen einfache API-Latenzen und verbinden sie anschliessend mit einer Quantisierungsplanung fuer verschiedene Geraeteklassen.

**Pflichtschritte:**
- Fuehre mehrere Requests mit zwei Temperaturprofilen aus.
- Vergleiche Laufzeit und Tokenverbrauch.
- Berechne danach den Speicherbedarf verschiedener Modellgroessen und Bit-Tiefen.

**Soll-Ergebnis:** eine kleine Entscheidungsgrundlage fuer Deployment auf begrenzter Hardware.

In [ ]:
benchmark_prompts = [
    {"label": "kurz", "prompt": "Erklaere RAG in einem Satz."},
    {"label": "mittel", "prompt": "Nenne zwei Vorteile von Embeddings gegenueber reiner Keyword-Suche."},
    {"label": "lang", "prompt": "Erklaere kurz drei Kriterien, mit denen man ein RAG-System bewerten kann."},
]
temperatures = [0.0, 0.7]

benchmark_rows = []
for temperature in temperatures:
    for item in benchmark_prompts:
        result = client.get(
            "/generate",
            params={"prompt": item["prompt"], "temperature": temperature},
        ).json()
        benchmark_rows.append(
            {
                "profil": f"temp={temperature}",
                "prompt": item["label"],
                "latency_s": result["latency_s"],
                "prompt_tokens": result["prompt_eval_count"],
                "generation_tokens": result["eval_count"],
            }
        )

print("Serving-Benchmark:")
for row in benchmark_rows:
    print(
        f"- {row['profil']} | {row['prompt']} | latency={row['latency_s']}s | "
        f"prompt_tokens={row['prompt_tokens']} | generation_tokens={row['generation_tokens']}"
    )


def calc_mem_gb(params_billion, bits):
    return (params_billion * bits) / 8


device_limits = {
    "8GB Laptop": 8,
    "16GB Workstation": 16,
    "32GB Server": 32,
}

print("\nSpeicherplanung:")
for params_billion in [0.8, 7, 13]:
    for bits in [16, 8, 4]:
        mem_gb = round(calc_mem_gb(params_billion, bits), 2)
        fits = [device for device, limit in device_limits.items() if mem_gb <= limit]
        fit_text = ", ".join(fits) if fits else "kein Geraet in der Liste"
        print(f"- {params_billion}B @ {bits}-bit -> ca. {mem_gb} GB | passt auf: {fit_text}")

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Vergleiche die beiden Modelfile-Profile an derselben Eingabe.
2. Dokumentiere die API-Antwortstruktur und mindestens zwei Benchmark-Zeilen.
3. Begruende in 3 bis 5 Saetzen, welches Deployment-Profil du fuer einen 8GB-Laptop waehlen wuerdest.

**Transferaufgaben:**
1. Erweitere die API um einen POST-Endpunkt mit JSON-Body.
2. Fuege ein zusaetzliches Sanitizing fuer Kreditkartennummern hinzu.
3. Teste ein weiteres Modellprofil und vergleiche dessen Latenz mit den vorhandenen Profilen.